# WIP.  A bunch of one-off functions overlap one another in some functionality aspects.

In [23]:
from urllib.parse import urljoin
from bs4 import BeautifulSoup

import pathlib
from pathlib import Path # get rid of that
import geopandas as gpd
import glob
import pandas as pd

import requests, logging
from bs4 import BeautifulSoup
import os
import zipfile
from urllib.parse import urljoin
import warnings
from collections import OrderedDict
import tempfile # Added import
import os # Needed for path joining within temp dir

In [24]:
def process_geopackages_from_zips(directory_path: str | Path) -> gpd.GeoDataFrame | None:
    """
    Extracts GeoPackage files, attempts to read 'buildings' layer, falls back
    to the first layer if 'buildings' is not found, handles encoding issues,
    attempts reprojection (skipping files that fail), combines results,
    and reports statistics.

    Args:
        directory_path: The path to the directory containing zip archives.

    Returns:
        The combined GeoDataFrame reprojected to EPSG:4326, or None if no
        GeoPackages were successfully processed or an error occurred during
        combination.
    """
    # Configure logging
    logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s', force=True)

    target_dir = Path(directory_path)
    if not target_dir.is_dir():
        logging.error(f"Directory not found at {directory_path}")
        return None

    zip_files = list(target_dir.glob('*.zip'))
    if not zip_files:
        logging.warning(f"No zip files found in {target_dir}.")
        return None

    logging.info(f"Found {len(zip_files)} zip files in {target_dir}.")

    all_gdfs = [] # For successfully reprojected GDFs
    total_gpkg_size = 0
    processed_count = 0 # Count files successfully read and reprojected
    errors = [] # For all errors encountered (extract, read, reproject)

    for zip_path in zip_files:
        gpkg_member_info = None
        gpkg_members_found = []
        gdf_read_success = False # Flag to track if reading succeeded
        current_gdf = None # To store the successfully read GDF
        gpkg_path_in_zip = 'N/A' # Default for error reporting
        read_layer_name = None # Track which layer was actually read

        try:
            with zipfile.ZipFile(zip_path, 'r') as zf:
                gpkg_members_found = [
                    m for m in zf.infolist()
                    if not m.is_dir() and Path(m.filename).suffix.lower() == '.gpkg'
                ]

                if len(gpkg_members_found) == 1:
                    gpkg_member_info = gpkg_members_found[0]
                    gpkg_path_in_zip = gpkg_member_info.filename # Update for specific file name
                    logging.info(f"Processing '{gpkg_path_in_zip}' in '{zip_path.name}' (Size: {gpkg_member_info.file_size / (1024*1024):.2f} MB)")
                    total_gpkg_size += gpkg_member_info.file_size

                    try:
                        with tempfile.TemporaryDirectory() as temp_dir:
                            zf.extract(gpkg_member_info, path=temp_dir)
                            extracted_file_path = Path(temp_dir) / gpkg_path_in_zip
                            logging.debug(f"Extracted '{gpkg_path_in_zip}' to temporary path: {extracted_file_path}")

                            # --- Layer Reading Logic (without fiona.listlayers) ---
                            target_layer_name = 'geoai_buildings'
                            current_gdf = None
                            gdf_read_success = False

                            try:
                                # Attempt 1: Read 'buildings' layer directly
                                try:
                                    current_gdf = gpd.read_file(extracted_file_path, layer=target_layer_name)
                                    logging.debug(f"Read layer '{target_layer_name}' from {extracted_file_path} with default encoding.")
                                    gdf_read_success = True
                                    read_layer_name = target_layer_name
                                except UnicodeDecodeError:
                                    logging.warning(f"UTF-8 decode failed for layer '{target_layer_name}' in {extracted_file_path}. Retrying with 'latin1' encoding.")
                                    try:
                                        current_gdf = gpd.read_file(extracted_file_path, layer=target_layer_name, encoding='latin1')
                                        logging.debug(f"Read layer '{target_layer_name}' from {extracted_file_path} with 'latin1' encoding.")
                                        gdf_read_success = True
                                        read_layer_name = target_layer_name
                                    except Exception as read_e_latin1:
                                        # Log if latin1 fails, but still proceed to ValueError check below
                                        logging.error(f"GeoPandas Read Error (layer: '{target_layer_name}', encoding: latin1): {read_e_latin1}")
                                        # Fall through to the ValueError check below

                                # Check if read was successful before moving to except block
                                if gdf_read_success:
                                     pass # Proceed to reprojection later

                            except ValueError as ve:
                                # Check if ValueError indicates the layer doesn't exist
                                # This is heuristic; a more robust check might inspect ve.args
                                if 'not found' in str(ve).lower() or 'does not exist' in str(ve).lower():
                                    logging.warning(f"Layer '{target_layer_name}' not found in {gpkg_path_in_zip}. Attempting to read the first available layer.")
                                    try:
                                        # Attempt 2: Read the first available layer (default)
                                        try:
                                            current_gdf = gpd.read_file(extracted_file_path)
                                            # We don't know the first layer's name here easily
                                            logging.debug(f"Read first available layer from {extracted_file_path} with default encoding.")
                                            gdf_read_success = True
                                            read_layer_name = "[First Available]" # Placeholder name
                                            logging.info(f"Successfully read the first available layer from {gpkg_path_in_zip} (since '{target_layer_name}' was not found).")
                                        except UnicodeDecodeError:
                                            logging.warning(f"UTF-8 decode failed for the first layer in {extracted_file_path}. Retrying with 'latin1' encoding.")
                                            try:
                                                current_gdf = gpd.read_file(extracted_file_path, encoding='latin1')
                                                logging.debug(f"Read first available layer from {extracted_file_path} with 'latin1' encoding.")
                                                gdf_read_success = True
                                                read_layer_name = "[First Available]" # Placeholder name
                                                logging.info(f"Successfully read the first available layer from {gpkg_path_in_zip} with 'latin1' encoding (since '{target_layer_name}' was not found).")
                                            except Exception as read_first_latin1_e:
                                                err_msg = f"GeoPandas Read Error (first layer, encoding: latin1): {read_first_latin1_e}"
                                                errors.append((zip_path.name, gpkg_path_in_zip, err_msg))
                                                logging.error(f"Failed to read first layer with latin1 encoding from '{extracted_file_path}': {err_msg}")
                                                # gdf_read_success remains False

                                    except Exception as read_first_e:
                                        # This catches errors trying to read the *first* layer
                                        err_msg = f"GeoPandas Read Error (first layer after '{target_layer_name}' failed): {read_first_e}"
                                        errors.append((zip_path.name, gpkg_path_in_zip, err_msg))
                                        logging.error(f"Failed to read the first layer from '{extracted_file_path}': {err_msg}")
                                        # gdf_read_success remains False
                                else:
                                    # The ValueError was likely not about the layer name, but some other issue
                                    err_msg = f"GeoPandas Read Error (layer: '{target_layer_name}'): {ve}"
                                    errors.append((zip_path.name, gpkg_path_in_zip, err_msg))
                                    logging.error(f"Failed to read layer '{target_layer_name}' from '{extracted_file_path}': {err_msg}")
                                    # gdf_read_success remains False

                            except Exception as read_e:
                                # Catch other unexpected errors during the initial attempt to read 'buildings'
                                err_msg = f"GeoPandas Read Error (layer: '{target_layer_name}'): {read_e}"
                                errors.append((zip_path.name, gpkg_path_in_zip, err_msg))
                                logging.error(f"Failed processing '{extracted_file_path}' from '{zip_path.name}': {err_msg}")
                                # gdf_read_success remains False

                    except Exception as extract_e:
                        err_msg = f"Error extracting file from zip: {extract_e}"
                        errors.append((zip_path.name, gpkg_path_in_zip, err_msg)) # Use gpkg_path_in_zip here
                        logging.error(f"Failed to extract '{gpkg_path_in_zip}' from '{zip_path.name}': {err_msg}")
                        # gdf_read_success remains False

                    # --- Attempt Reprojection ONLY if reading succeeded ---
                    if gdf_read_success and current_gdf is not None:
                        original_crs = current_gdf.crs
                        logging.info(f"Original CRS for '{gpkg_path_in_zip}' (Layer: {read_layer_name}): {original_crs}")

                        if not original_crs:
                            reproj_err_msg = f"Missing CRS (Layer: {read_layer_name}), cannot reproject accurately"
                            errors.append((zip_path.name, gpkg_path_in_zip, reproj_err_msg))
                            logging.warning(f"GPKG '{gpkg_path_in_zip}' in '{zip_path.name}' has no CRS. Skipping reprojection.")
                        else:
                            try:
                                reprojected_gdf = current_gdf.to_crs('EPSG:4326')
                                all_gdfs.append(reprojected_gdf)
                                processed_count += 1 # Increment successful processing count
                                logging.debug(f"Successfully reprojected '{gpkg_path_in_zip}' (Layer: {read_layer_name}).")
                            except Exception as reproj_e:
                                # Catch any reprojection error
                                err_type = type(reproj_e).__name__
                                err_msg = f"Reprojection Error ({err_type}) from '{original_crs}' (Layer: {read_layer_name}): {reproj_e}"
                                errors.append((zip_path.name, gpkg_path_in_zip, err_msg)) # Log detailed error
                                logging.error(f"Failed to reproject '{gpkg_path_in_zip}' from '{zip_path.name}': {err_msg}")
                                # Do NOT add to all_gdfs list
                    elif not gdf_read_success:
                         logging.debug(f"Skipping reprojection for '{gpkg_path_in_zip}' due to read failure.")


                elif len(gpkg_members_found) == 0:
                    errors.append((zip_path.name, 'N/A', "No .gpkg file found"))
                    logging.warning(f"No .gpkg file found in '{zip_path.name}'.")
                else:
                    found_files = [m.filename for m in gpkg_members_found]
                    err_msg = f"Multiple .gpkg files found: {', '.join(found_files)}"
                    errors.append((zip_path.name, ', '.join(found_files), err_msg))
                    logging.warning(f"Multiple .gpkg files found in '{zip_path.name}': {found_files}. Skipping archive.")

        except zipfile.BadZipFile:
            errors.append((zip_path.name, 'N/A', "Bad zip file or corrupt archive"))
            logging.error(f"Could not open '{zip_path.name}', it may be corrupt.")
        except FileNotFoundError:
             errors.append((zip_path.name, 'N/A', "Zip file not found during processing (unexpected)"))
             logging.error(f"File not found '{zip_path.name}' during processing loop.")
        except Exception as e:
            # Try to determine the gpkg file involved if possible
            current_gpkg_ref = gpkg_path_in_zip # Will be 'N/A' if not found/extracted yet
            if current_gpkg_ref == 'N/A' and gpkg_members_found:
                 current_gpkg_ref = ', '.join(m.filename for m in gpkg_members_found)

            err_msg = f"Unexpected Error during zip processing: {e}"
            errors.append((zip_path.name, current_gpkg_ref, err_msg))
            logging.error(f"An unexpected error occurred processing '{zip_path.name}': {e}")


    # --- Summary Reporting ---
    logging.info("\n--- Processing Summary ---")
    zips_with_errors = set(err[0] for err in errors)
    logging.info(f"Total zip files processed: {len(zip_files)}")
    logging.info(f"Successfully read layer ('buildings' or first available) and reprojected to EPSG:4326: {processed_count} GeoPackage file(s)")
    logging.info(f"Zip archives with errors (extract/read/reproject): {len(zips_with_errors)}")
    logging.info(f"Total size of identified GeoPackage members (original compressed): {total_gpkg_size / (1024*1024):.2f} MB")

    if errors:
        logging.warning("\n--- Errors Encountered During Processing ---")
        errors.sort(key=lambda x: x[0]) # Sort by zip file name
        for zip_name, gpkg_name, error_msg in errors:
            logging.warning(f"- Archive: '{zip_name}', File Ref: '{gpkg_name}', Issue: {error_msg}")
    else:
        logging.info("\nNo errors encountered during processing.")

    # --- Combine Successfully Processed GeoDataFrames ---
    combined_gdf = None
    if not all_gdfs:
        logging.warning("\nNo GeoDataFrames were successfully processed and reprojected, cannot combine.")
    else:
        logging.info(f"\nCombining {len(all_gdfs)} successfully processed GeoDataFrames...")
        try:
            # Pre-concat CRS check (optional but good practice)
            for i, gdf in enumerate(all_gdfs):
                if gdf.crs != 'EPSG:4326':
                    logging.warning(f"Warning: GDF index {i} has unexpected CRS {gdf.crs} before concat. Forcing EPSG:4326.")
                    try:
                       all_gdfs[i] = gdf.set_crs('EPSG:4326', allow_override=True)
                    except Exception as crs_err:
                        logging.error(f"Failed to force CRS on GDF index {i}: {crs_err}. Problems might occur during concatenation.")

            combined_gdf = pd.concat(all_gdfs, ignore_index=True)

            # Post-concat CRS check
            if combined_gdf.crs is None:
                 logging.warning("Combined GeoDataFrame CRS is None after concatenation. Setting to EPSG:4326.")
                 combined_gdf = combined_gdf.set_crs('EPSG:4326')
            elif combined_gdf.crs != 'EPSG:4326':
                 logging.error(f"Internal Error: Combined GDF CRS is unexpectedly {combined_gdf.crs} after concat. Attempting override.")
                 try:
                     combined_gdf = combined_gdf.set_crs('EPSG:4326', allow_override=True)
                     logging.info("Combination successful. Final CRS: EPSG:4326 (overridden).")
                 except Exception:
                     logging.error("Failed to override CRS on combined GDF. Returning potentially incorrect data or None.")
                     combined_gdf = None # Set to None as forcing failed
            else:
                 logging.info("Combination successful. Final CRS: EPSG:4326.")

        except Exception as concat_e:
            logging.error(f"\nError during final concatenation: {concat_e}")
            logging.error("This can happen if successfully processed GeoDataFrames have incompatible schemas.")
            combined_gdf = None # Ensure None is returned on concat error

    return combined_gdf

In [25]:
def download_files(url, output_dir):
  downloaded_count = 0
  download_errors = 0
  unzip_errors = 0
  error_files = {
    'download_errors': [],
    'unzip_errors': []
  }
  
  response = requests.get(url, verify=False)
  soup = BeautifulSoup(response.text, 'html.parser')
  
  for link in soup.find_all('a'):
    href = link.get('href')
    if href and href.endswith(('.zip')):
      file_url = urljoin(url, href)
      file_path = os.path.join(output_dir, href)
      
      # Download file with error handling
      try:
        response = requests.get(file_url, stream=True, verify=False)
        if response.status_code == 200:
          with open(file_path, 'wb') as f:
            for chunk in response.iter_content(chunk_size=1024):
              f.write(chunk)
          print(f'Downloaded {href}')
          downloaded_count += 1
          
          # Extract the zip file with error handling
          try:
            with zipfile.ZipFile(file_path, 'r') as zip_ref:
              zip_ref.extractall(os.path.dirname(file_path))
            print(f'Extracted {href}')
          except Exception as e:
            unzip_errors += 1
            error_files['unzip_errors'].append(href)
            print(f'Error extracting {href}: {str(e)}')
        else:
          download_errors += 1
          error_files['download_errors'].append(href)
          print(f'Error downloading {href}: HTTP status {response.status_code}')
      except Exception as e:
        download_errors += 1
        error_files['download_errors'].append(href)
        print(f'Error downloading {href}: {str(e)}')
  
  # Print summary
  print(f"\n📊 Download Summary:")
  print(f"Total zip files downloaded: {downloaded_count}")
  print(f"Download errors: {download_errors}")
  print(f"Extraction errors: {unzip_errors}")
  
  if download_errors > 0:
    print(f"\nFiles with download errors:")
    for file in error_files['download_errors']:
      print(f" - {file}")
      
  if unzip_errors > 0:
    print(f"\nFiles with extraction errors:")
    for file in error_files['unzip_errors']:
      print(f" - {file}")
  
  return {
    'downloaded': downloaded_count,
    'download_errors': download_errors,
    'unzip_errors': unzip_errors,
    'error_files': error_files
  }

In [26]:
def combine_shapefiles(output_dir, crs):
    dfs = []
    
    # Initialize processing counters
    processing_stats = {
        'total_files': 0,
        'complete_success': 0,
        'partial_success': 0,
        'complete_failure': 0,
        'unopenable_files': 0,
        'total_input_geometries': 0,
        'empty_geometries_removed': 0,
        'invalid_geometries_identified': 0,
        'geometries_fixed_by_buffer': 0,
        'geometries_unfixable': 0,
        'final_valid_geometries': 0
    }
    
    # Initialize detailed report dictionary
    detailed_report = {
        'file_reports': {},
        'error_files': [],
        'partial_success_files': [],
        'complete_success_files': [],
        'geometry_modifications': {}
    }
    
    import warnings
    
    with warnings.catch_warnings(record=True) as caught_warnings:
        warnings.simplefilter("always")
        
        try:
            for folder in pathlib.Path(output_dir).iterdir():
                if folder.is_dir():
                    shp_files = glob.glob(str(folder / '*.shp'))
                    if shp_files:
                        for shp_file in shp_files:
                            processing_stats['total_files'] += 1
                            warning_count = 0
                            
                            try:
                                caught_warnings[:] = []
                                df = gpd.read_file(shp_file)
                                warning_count = sum(1 for w in caught_warnings 
                                                 if "organizePolygons() received an unexpected geometry" in str(w.message))
                                
                                initial_count = len(df)
                                processing_stats['total_input_geometries'] += initial_count
                                
                                file_stats = {
                                    'total_geometries': initial_count,
                                    'empty_geometries': 0,
                                    'invalid_geometries': 0,
                                    'fixed_geometries': 0,
                                    'unfixable_geometries': 0,
                                    'warning_geometries': warning_count,
                                    'final_valid_geometries': 0,
                                    'status': 'success'
                                }
                                
                                # Step 1: Remove empty/null geometries
                                empty_mask = df.geometry.isna()
                                empty_count = empty_mask.sum()
                                file_stats['empty_geometries'] = empty_count
                                processing_stats['empty_geometries_removed'] += empty_count
                                
                                df = df[~empty_mask]
                                
                                # Step 2: Identify invalid geometries
                                invalid_mask = ~df.geometry.is_valid
                                invalid_count = invalid_mask.sum()
                                file_stats['invalid_geometries'] = invalid_count
                                processing_stats['invalid_geometries_identified'] += invalid_count
                                
                                # Create a copy of invalid geometries for the detailed report
                                if invalid_count > 0:
                                    invalid_df = df[invalid_mask].copy()
                                    detailed_report['geometry_modifications'][shp_file] = []
                                    
                                    # Store the "before" state for each invalid geometry
                                    for idx, row in invalid_df.iterrows():
                                        # Use WKT representation as it's more human-readable
                                        before_wkt = row.geometry.wkt if row.geometry else None
                                        
                                        # Step 3: Apply buffer(0) to fix geometries
                                        fixed_geom = row.geometry.buffer(0) if row.geometry else None
                                        after_wkt = fixed_geom.wkt if fixed_geom else None
                                        
                                        # Record the modification
                                        detailed_report['geometry_modifications'][shp_file].append({
                                            'id': idx,
                                            'before': before_wkt,
                                            'after': after_wkt,
                                            'was_fixed': fixed_geom.is_valid if fixed_geom else False,
                                            'geometry_type': row.geometry.geom_type if row.geometry else None
                                        })
                                
                                # Apply the fix to the main dataframe
                                df['original_valid'] = df.geometry.is_valid
                                df['original_geometry'] = df.geometry
                                
                                # Step 3: Apply buffer(0) to try to fix invalid geometries
                                df['geometry'] = df.apply(
                                    lambda row: row.geometry.buffer(0) if not row.geometry.is_valid else row.geometry, 
                                    axis=1
                                )
                                
                                # Count fixed geometries
                                fixed_geometries = sum(
                                    (~df['original_valid']) & (df.geometry.is_valid)
                                )
                                file_stats['fixed_geometries'] = fixed_geometries
                                processing_stats['geometries_fixed_by_buffer'] += fixed_geometries
                                
                                # Step 4: Filter out geometries that remained invalid after repair
                                still_invalid_mask = ~df.geometry.is_valid
                                still_invalid_count = still_invalid_mask.sum()
                                file_stats['unfixable_geometries'] = still_invalid_count
                                processing_stats['geometries_unfixable'] += still_invalid_count
                                
                                df = df[df.geometry.is_valid]
                                
                                # Clean up temporary columns
                                if 'original_valid' in df.columns:
                                    df = df.drop(columns=['original_valid', 'original_geometry'])
                                
                                final_count = len(df)
                                file_stats['final_valid_geometries'] = final_count
                                processing_stats['final_valid_geometries'] += final_count
                                
                                if final_count == 0:
                                    file_stats['status'] = 'complete_failure'
                                    processing_stats['complete_failure'] += 1
                                    detailed_report['error_files'].append(shp_file)
                                elif empty_count + invalid_count + warning_count == 0:
                                    file_stats['status'] = 'complete_success'
                                    processing_stats['complete_success'] += 1
                                    detailed_report['complete_success_files'].append(shp_file)
                                else:
                                    file_stats['status'] = 'partial_success'
                                    processing_stats['partial_success'] += 1
                                    detailed_report['partial_success_files'].append(shp_file)
                                
                                detailed_report['file_reports'][shp_file] = file_stats
                                
                                if len(df) > 0:
                                    # Convert to target CRS
                                    df = df.to_crs(crs)
                                    
                                    # # Create the bbox column with [xmin, ymin, xmax, ymax] for each geometry
                                    # bounds = df.geometry.bounds
                                    # df['bbox'] = [
                                    #     [xmin, ymin, xmax, ymax] 
                                    #     for xmin, ymin, xmax, ymax in zip(
                                    #         bounds['minx'], bounds['miny'], 
                                    #         bounds['maxx'], bounds['maxy']
                                    #     )
                                    # ]
                                    
                                    dfs.append(df)
                                    
                            except Exception as e:
                                processing_stats['unopenable_files'] += 1
                                detailed_report['file_reports'][shp_file] = {
                                    'error': str(e),
                                    'total_geometries': 0,
                                    'empty_geometries': 0,
                                    'invalid_geometries': 0,
                                    'fixed_geometries': 0,
                                    'unfixable_geometries': 0,
                                    'warning_geometries': 0,
                                    'final_valid_geometries': 0,
                                    'status': 'complete_failure'
                                }
                                processing_stats['complete_failure'] += 1
                                detailed_report['error_files'].append(shp_file)
            
            if not dfs:
                raise ValueError("No valid data found in any of the shapefiles")
            
            # Combine all dataframes
            combined_df = pd.concat(dfs, ignore_index=True)
            
            # Print only essential statistics
            print("\n📊 Overall Processing Statistics:")
            print(f"Total files processed: {processing_stats['total_files']}")
            print(f"✅ Complete success: {processing_stats['complete_success']} files ({(processing_stats['complete_success']/processing_stats['total_files'])*100:.1f}%)")
            print(f"⚠️  Partial success: {processing_stats['partial_success']} files ({(processing_stats['partial_success']/processing_stats['total_files'])*100:.1f}%)")
            print(f"❌ Complete failure: {processing_stats['complete_failure']} files ({(processing_stats['complete_failure']/processing_stats['total_files'])*100:.1f}%)")
            
            print("\n📈 Final Summary:")
            total_geoms = processing_stats['total_input_geometries']
            empty_geoms = processing_stats['empty_geometries_removed']
            invalid_geoms = processing_stats['invalid_geometries_identified']
            fixed_geoms = processing_stats['geometries_fixed_by_buffer']
            unfixable_geoms = processing_stats['geometries_unfixable']
            final_valid_geoms = processing_stats['final_valid_geometries']
            
            print(f"Total geometries across all files: {total_geoms}")
            print(f"Empty geometries removed: {empty_geoms}")
            print(f"Invalid geometries identified: {invalid_geoms}")
            print(f"Invalid geometries fixed by buffer(0): {fixed_geoms}")
            print(f"Geometries that couldn't be fixed: {unfixable_geoms}")
            print(f"Total geometries lost: {total_geoms - final_valid_geoms}")
            print(f"Total geometries in final output: {final_valid_geoms}")
            
            if processing_stats['unopenable_files'] > 0:
                print(f"Number of unopenable files: {processing_stats['unopenable_files']}")
            
            retention_rate = (final_valid_geoms / total_geoms * 100) if total_geoms > 0 else 0
            print(f"Overall geometry retention rate: {retention_rate:.1f}%")
            
            return combined_df, processing_stats, detailed_report
            
        except Exception as e:
            print(f"An unexpected error occurred: {str(e)}")
            raise

# GeoAI

In [27]:
geoai_url = 'https://ftp.maps.canada.ca/pub/nrcan_rncan/vector/geobase_geoai_geoia/GPKG/'

In [28]:
geoai_input_dir = "geoai_buildings"


In [29]:
geoai_output_dir = "geoai_buildings"

In [20]:
if not os.path.exists(geoai_output_dir):
  os.makedirs(geoai_output_dir)

In [21]:
geoai_parquet_1_1_wkb_output_file = pathlib.Path("geoai_buildings_4326-wkb-1.1.parquet")


In [ ]:
download_files(geoai_url, geoai_output_dir)


In [ ]:
combined_gdf = process_geopackages_from_zips(geoai_input_dir)

In [ ]:
combined_gdf

In [31]:
combined_gdf.to_parquet(geoai_parquet_1_1_wkb_output_file, write_covering_bbox=True)


# Stats Can ODB v3

In [3]:
odb_input_dir = "buildings_stats_can"

In [9]:
odb_parquet_1_1_wkb_output_file = pathlib.Path("buildings_stats_can_4326-wkb-1.1.parquet")

In [ ]:
download_files(geoai_url, geoai_output_dir)


In [ ]:
combined_gdf = process_geopackages_from_zips(odb_input_dir)

In [ ]:
combined_gdf

In [10]:
combined_gdf.to_parquet(odb_parquet_1_1_wkb_output_file, write_covering_bbox=True)

In [ ]:
len(combined_gdf['dataset'].unique())


In [ ]:
buildings_df.to_parquet(parquet_1_1_wkb_output_file, write_covering_bbox=True)

In [17]:
autobuildings_url = 'https://ftp.maps.canada.ca/pub/nrcan_rncan/extraction/auto_building/shp/'
open_database_buildings_url = 'https://www150.statcan.gc.ca/pub/34-26-0001/2018001/zip/'

In [18]:
output_dir_autobuildings = 'buildings_shp'
outputdir_open_database_buildings = 'buildings_stats_can'

In [19]:
url = open_database_buildings_url
output_dir = outputdir_open_database_buildings

In [20]:
if not os.path.exists(output_dir):
  os.makedirs(output_dir)

In [ ]:
download_files(url, output_dir)

In [ ]:
buildings_df, stats, detailed_report = combine_shapefiles(output_dir, 'EPSG:4326')

In [ ]:
stats

In [ ]:
detailed_report

In [32]:
gpkg_output_file = pathlib.Path("canada-building-footprints-4326.gpkg")
parquet_arrow_output_file = pathlib.Path("canada-building-footprints-4326-arrow.parquet") # Arrow encoding not supported by QGIS plugin; only wkb
parquet_wkb_output_file = pathlib.Path("canada-building-footprints-4326-wkb.parquet")
parquet_1_1_wkb_output_file = pathlib.Path("canada-building-footprints-4326-wkb-1.1.parquet")

In [49]:
buildings_df.to_file(gpkg_output_file, driver="GPKG")

In [50]:
# geoarrow encoding not supported in QGIS 3.40
buildings_df.to_parquet(parquet_arrow_output_file, geometry_encoding='geoarrow')

In [33]:
buildings_df.to_parquet(parquet_1_1_wkb_output_file, write_covering_bbox=True)

In [ ]:
buildings_df.to_parquet(parquet_1_1_wkb_output_file, write_covering_bbox=True)

In [ ]:
buildings_df.sindex

In [ ]:
buildings_df.head